# Load data

In [68]:
import pandas as pd
df = df = pd.read_csv("quality_data.csv")
print(df.head())

                narration  mode    type    category  subcategory
0    FD interest credited  NEFT  Credit  investment  fd_interest
1      FD interest payout  NEFT  Credit  investment  fd_interest
2  Fixed deposit interest  NEFT  Credit  investment  fd_interest
3          Interest on FD  NEFT  Credit  investment  fd_interest
4    FD maturity interest  NEFT  Credit  investment  fd_interest


# Create input_text

In [69]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [70]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [71]:
df["label"] = df["category"] + "_" + df["subcategory"]

In [72]:
print(df[["input_text", "label"]].head())

                                          input_text                   label
0  narration: fd interest credited | mode: neft |...  investment_fd_interest
1  narration: fd interest payout | mode: neft | t...  investment_fd_interest
2  narration: fixed deposit interest | mode: neft...  investment_fd_interest
3  narration: interest on fd | mode: neft | type:...  investment_fd_interest
4  narration: fd maturity interest | mode: neft |...  investment_fd_interest


In [73]:
from datasets import Dataset

df["input_text"] = (
    df["narration"] + " " + df["mode"] + " " + df["type"]
)

df["input_text"] = df["input_text"].str.lower()

dataset = Dataset.from_pandas(df[["input_text", "label"]])

In [74]:
labels = list(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode)

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [75]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [76]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [77]:
import sys
!{sys.executable} -m pip install --upgrade accelerate transformers torch


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  supe

Step,Training Loss
10,2.385914
20,2.338143
30,2.096492
40,1.791595
50,1.673269
60,1.378238
70,1.181476
80,1.066917
90,1.031524
100,0.946589


TrainOutput(global_step=108, training_loss=1.535637621526365, metrics={'train_runtime': 767.9579, 'train_samples_per_second': 0.563, 'train_steps_per_second': 0.141, 'total_flos': 57235101106176.0, 'train_loss': 1.535637621526365, 'epoch': 6.0})

In [79]:
import torch
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    category, subcategory = label.split("_", 1)

    return category, subcategory, round(confidence, 2)

def confidence_level(conf):
    if conf >= 0.65:
        return "High"
    elif conf >= 0.4:
        return "Medium"
    else:
        return "Low"

In [80]:
results = []

for i in range(len(df)):
    narration = df["narration"].iloc[i]
    mode = df["mode"].iloc[i]
    txn_type = df["type"].iloc[i]

    # same input used for model
    text = df["input_text"].iloc[i]

    pred_category, pred_subcategory, confidence = predict(text)

    results.append({
        "narration": narration,
        "mode": mode,
        "type": txn_type,
        "category": df["category"].iloc[i],
        "subcategory": df["subcategory"].iloc[i],
        "predicted_category": pred_category,
        "predicted_subcategory": pred_subcategory,
        "confidence": confidence,
        "confidence_percent": f"{round(confidence*100,1)}%",
        "confidence_level": confidence_level(confidence),
        "correct": (df["category"].iloc[i] == pred_category) and 
           (df["subcategory"].iloc[i] == pred_subcategory)
    })

result_df = pd.DataFrame(results)

print(result_df.to_string())

                        narration        mode    type    category  subcategory predicted_category predicted_subcategory  confidence confidence_percent confidence_level  correct
0            FD interest credited        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.56              56.0%           Medium     True
1              FD interest payout        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.51              51.0%           Medium     True
2          Fixed deposit interest        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.52              52.0%           Medium     True
3                  Interest on FD        NEFT  Credit  investment  fd_interest         investment           fd_interest        0.44              44.0%           Medium     True
4            FD maturity interest        NEFT  Credit  investment  fd_interest         investment           fd_inte

## | Metric | Meaning      | Use here               |
### | ------ | ------------ | ---------------------- |
### | F1     | balance      | good baseline          |
### | F2     | recall heavy | better for your system |


In [81]:
y_true = result_df["category"] + "_" + result_df["subcategory"]
y_pred = result_df["predicted_category"] + "_" + result_df["predicted_subcategory"]

In [82]:
from sklearn.metrics import f1_score, fbeta_score

f1 = f1_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Overall Metrics:")
print(f"F1 Score: {round(f1,3)}")
print(f"F2 Score: {round(f2,3)}")

Overall Metrics:
F1 Score: 0.91
F2 Score: 0.91


In [83]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print(report_df)

                        precision    recall  f1-score    support
expense_bills            0.750000  0.750000  0.750000   8.000000
expense_food             0.875000  0.875000  0.875000   8.000000
expense_health           1.000000  0.571429  0.727273   7.000000
expense_insurance        1.000000  1.000000  1.000000   8.000000
expense_loan             1.000000  1.000000  1.000000   8.000000
expense_shopping         0.875000  0.875000  0.875000   8.000000
expense_transport        0.769231  1.000000  0.869565  10.000000
income_salary            1.000000  1.000000  1.000000   8.000000
investment_fd_booking    1.000000  1.000000  1.000000   8.000000
investment_fd_interest   1.000000  0.900000  0.947368  10.000000
investment_stocks        0.888889  1.000000  0.941176   8.000000
accuracy                 0.912088  0.912088  0.912088   0.912088
macro avg                0.923465  0.906494  0.907762  91.000000
weighted avg             0.920917  0.912088  0.909776  91.000000


In [84]:
from sklearn.metrics import fbeta_score

labels = list(set(y_true))

f2_scores = {}

for label in labels:
    y_true_bin = [1 if y == label else 0 for y in y_true]
    y_pred_bin = [1 if y == label else 0 for y in y_pred]

    f2_scores[label] = fbeta_score(y_true_bin, y_pred_bin, beta=2)

report_df["f2_score"] = report_df.index.map(f2_scores)

In [85]:
report_df = report_df.round(3)
print(report_df)

                        precision  recall  f1-score  support  f2_score
expense_bills               0.750   0.750     0.750    8.000     0.750
expense_food                0.875   0.875     0.875    8.000     0.875
expense_health              1.000   0.571     0.727    7.000     0.625
expense_insurance           1.000   1.000     1.000    8.000     1.000
expense_loan                1.000   1.000     1.000    8.000     1.000
expense_shopping            0.875   0.875     0.875    8.000     0.875
expense_transport           0.769   1.000     0.870   10.000     0.943
income_salary               1.000   1.000     1.000    8.000     1.000
investment_fd_booking       1.000   1.000     1.000    8.000     1.000
investment_fd_interest      1.000   0.900     0.947   10.000     0.918
investment_stocks           0.889   1.000     0.941    8.000     0.976
accuracy                    0.912   0.912     0.912    0.912       NaN
macro avg                   0.923   0.906     0.908   91.000       NaN
weight

In [86]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_insurance 1.0

Worst Performing Category:
expense_health 0.727
